In [1]:
!pip install transformers torch


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from transformers import pipeline

C:\Users\siddh\TruthLens\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
nli_model = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli"
)

In [ ]:
def verify_claim(claim, evidence):

    result = nli_model(
        f"{claim} </s></s> {evidence}"
    )[0]

    label = result['label']
    score = result['score']

    return label, round(score, 3)

In [ ]:
def verify_with_evidence(claim, evidences):

    results = []

    for e in evidences:
        label, score = verify_claim(claim, e)
        results.append((e, label, score))

    return results

In [ ]:
def final_decision(results):

    support = 0
    contradict = 0
    neutral = 0

    for _, label, _ in results:
        if label == "ENTAILMENT":
            support += 1
        elif label == "CONTRADICTION":
            contradict += 1
        else:
            neutral += 1

    if support > contradict and support > neutral:
        return "SUPPORTED"
    elif contradict > support and contradict > neutral:
        return "CONTRADICTED"
    else:
        return "NEUTRAL"

In [ ]:
def verify_pipeline(claim):

    evidences = hybrid_retrieve(claim, use_mock=False)

    verification_results = verify_with_evidence(claim, evidences)

    decision = final_decision(verification_results)

    return {
        "claim": claim,
        "decision": decision,
        "details": verification_results
    }

In [ ]:
for claim in claims:

    output = verify_pipeline(claim)

    print("\n==============================")
    print("CLAIM:", output["claim"])
    print("FINAL:", output["decision"])

    print("\nDetails:")
    for e, label, score in output["details"]:
        print(f"{label} ({score}) → {e}")